# Backtest Metrics Deep Dive

`BacktestResult` computes **19 performance metrics** from the trade history.
This notebook explains each metric, shows how it's computed, and
demonstrates how to analyse backtest results.

## Metrics at a Glance

| Category | Metrics |
|----------|--------|
| **Return** | `total_return`, `final_cash`, `avg_pnl` |
| **Win/Loss** | `total_trades`, `winning_trades`, `losing_trades`, `win_rate` |
| **Risk** | `max_drawdown`, `sharpe_ratio`, `recovery_factor` |
| **Quality** | `profit_factor`, `expectancy`, `avg_win`, `avg_loss` |
| **Streaks** | `max_consecutive_wins`, `max_consecutive_losses` |
| **Costs** | `total_commission`, `total_slippage` |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from quaver.backtest.engine import BacktestEngine
from quaver.backtest.portfolio import CommissionConfig, ExitRules, Portfolio, SlippageConfig
from quaver.strategies.base import BaseStrategy, SignalOutput
from quaver.types import SignalDirection

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["figure.dpi"] = 100

## Generate a Backtest with Many Trades

In [ ]:
# Oscillating price -> lots of mean reversion trades
np.random.seed(42)
n = 400
base = 100 + 12 * np.sin(np.linspace(0, 12 * np.pi, n)) + np.cumsum(np.random.randn(n) * 0.3)
high = base + np.abs(np.random.randn(n) * 1.5)
low = base - np.abs(np.random.randn(n) * 1.5)
rows = []
for i in range(n):
    rows.append(
        {
            "ts": datetime(2023, 1, 1) + timedelta(days=i),
            "open": base[i] + np.random.randn() * 0.3,
            "high": max(high[i], base[i]),
            "low": min(low[i], base[i]),
            "close": base[i],
            "volume": 1_000_000.0,
        }
    )
candles = pd.DataFrame(rows)


class SwingTrader(BaseStrategy):
    def validate_parameters(self):
        pass

    def get_required_candle_count(self):
        return 20

    def compute(self, candles, as_of):
        closes = candles["close"].values
        ma = np.mean(closes[-20:])
        price = closes[-1]
        if price < ma * 0.97:
            return SignalOutput(direction=SignalDirection.BUY, confidence=0.8)
        elif price > ma * 1.03:
            return SignalOutput(direction=SignalDirection.SELL, confidence=0.8)
        return None


portfolio = Portfolio(
    initial_capital=10_000,
    quantity_per_trade=10,
    commission=CommissionConfig(fixed_per_trade=1.0, pct_of_notional=0.001),
    slippage=SlippageConfig(slippage_pct=0.0005),
    exit_rules=ExitRules(stop_loss_pct=0.05, take_profit_pct=0.08),
)
engine = BacktestEngine(
    strategy=SwingTrader({}),
    portfolio=portfolio,
    instrument_id="SYN",
)
result = engine.run(candles)
print(f"Total trades: {result.total_trades}")

## Full Summary

In [ ]:
summary = result.summary()
pd.DataFrame(list(summary.items()), columns=["Metric", "Value"])

## Equity Curve & Drawdown

In [ ]:
cpnl = np.array(result.cumulative_pnl)
peak = np.maximum.accumulate(cpnl)
drawdown = cpnl - peak

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]})

axes[0].plot(range(len(cpnl)), cpnl, linewidth=1.5, label="Cumulative P&L")
axes[0].plot(range(len(peak)), peak, linewidth=0.8, linestyle="--", color="grey", label="Peak")
axes[0].fill_between(range(len(cpnl)), cpnl, peak, alpha=0.15, color="red")
axes[0].axhline(0, color="grey", linewidth=0.5)
axes[0].set_ylabel("P&L ($)")
axes[0].set_title("Equity Curve")
axes[0].legend()

axes[1].fill_between(range(len(drawdown)), drawdown, 0, alpha=0.4, color="red")
axes[1].plot(range(len(drawdown)), drawdown, linewidth=0.8, color="red")
axes[1].set_ylabel("Drawdown ($)")
axes[1].set_xlabel("Trade #")
axes[1].set_title(f"Drawdown (Max: {result.max_drawdown * 100:.2f}% of capital)")

plt.tight_layout()
plt.show()

## Trade Analysis

In [ ]:
trades_df = pd.DataFrame(
    [
        {
            "entry": t.entry_ts.strftime("%Y-%m-%d"),
            "exit": t.exit_ts.strftime("%Y-%m-%d"),
            "direction": t.direction.value,
            "entry_price": round(t.entry_price, 2),
            "exit_price": round(t.exit_price, 2),
            "qty": t.quantity,
            "pnl": round(t.pnl, 2),
            "net_pnl": round(t.net_pnl, 2),
            "commission": round(t.commission, 2),
            "exit_reason": t.exit_reason.value if t.exit_reason else "n/a",
        }
        for t in result.trades
    ]
)
trades_df

## P&L Distribution

In [ ]:
pnl = np.array(result.pnl_series)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
colors = ["green" if p > 0 else "red" for p in pnl]
axes[0].bar(range(len(pnl)), pnl, color=colors, alpha=0.7, width=1.0)
axes[0].axhline(0, color="grey", linewidth=0.5)
axes[0].axhline(
    result.avg_pnl, color="blue", linestyle="--", linewidth=1, label=f"Mean: {result.avg_pnl:.2f}"
)
axes[0].set_title("Per-Trade P&L")
axes[0].set_xlabel("Trade #")
axes[0].set_ylabel("P&L ($)")
axes[0].legend()

# Distribution
axes[1].hist(pnl, bins=20, color="steelblue", edgecolor="white", alpha=0.8)
axes[1].axvline(0, color="grey", linewidth=0.5)
axes[1].axvline(result.avg_pnl, color="blue", linestyle="--", linewidth=1)
axes[1].set_title("P&L Distribution")
axes[1].set_xlabel("P&L ($)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## Exit Reason Breakdown

In [ ]:
from collections import Counter

reason_counts = Counter(t.exit_reason.value if t.exit_reason else "n/a" for t in result.trades)

reason_pnl = {}
for t in result.trades:
    key = t.exit_reason.value if t.exit_reason else "n/a"
    reason_pnl.setdefault(key, []).append(t.pnl)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = list(reason_counts.keys())
counts = list(reason_counts.values())
axes[0].bar(labels, counts, color=["red", "green", "orange", "grey", "blue"][: len(labels)])
axes[0].set_title("Trades by Exit Reason")
axes[0].set_ylabel("Count")

# Avg P&L by exit reason
avg_by_reason = {k: np.mean(v) for k, v in reason_pnl.items()}
colors = ["green" if v > 0 else "red" for v in avg_by_reason.values()]
axes[1].bar(avg_by_reason.keys(), avg_by_reason.values(), color=colors, alpha=0.7)
axes[1].axhline(0, color="grey", linewidth=0.5)
axes[1].set_title("Average P&L by Exit Reason")
axes[1].set_ylabel("Avg P&L ($)")

plt.tight_layout()
plt.show()

## Key Metric Formulas

| Metric | Formula |
|--------|--------|
| `total_return` | `(final_cash - initial_capital) / initial_capital` |
| `win_rate` | `winning_trades / total_trades` |
| `max_drawdown` | `(trough - peak) / initial_capital` (always <= 0) |
| `sharpe_ratio` | `mean(pnl) / std(pnl) * sqrt(252)` |
| `profit_factor` | `sum(wins) / abs(sum(losses))` |
| `expectancy` | `win_rate * avg_win + (1 - win_rate) * avg_loss` |
| `recovery_factor` | `total_return / abs(max_drawdown)` |
| `net_pnl` | `pnl - commission - slippage_cost` |